In [1]:
import random
import math
from PIL import Image, ImageDraw, ImageFont
import time
import math
import pandas as pd

# Helper functions
def is_valid(board, row, col, num, grid_size, subgrid_size):
    """Check if placing num in board[row][col] is valid."""
    for i in range(grid_size):
        # Check row
        if board[row][i] == num:
            return False
        # Check column
        if board[i][col] == num:
            return False
        # Check subgrid
        subgrid_row = subgrid_size * (row // subgrid_size)
        subgrid_col = subgrid_size * (col // subgrid_size)
        if board[subgrid_row + i // subgrid_size][subgrid_col + i % subgrid_size] == num:
            return False
    return True


def find_mrv_cell(board, grid_size, subgrid_size):
    """Find the empty cell with the smallest domain (MRV heuristic)."""
    min_domain = grid_size + 1  # Start with an impossible domain size
    mrv_cell = None

    for row in range(grid_size):
        for col in range(grid_size):
            if board[row][col] == 0:  # Only consider empty cells
                # Calculate domain size for this cell
                domain = [num for num in range(1, grid_size + 1) if is_valid(board, row, col, num, grid_size, subgrid_size)]
                if len(domain) < min_domain:
                    min_domain = len(domain)
                    mrv_cell = (row, col)

    return mrv_cell


def get_lcv_order(board, row, col, grid_size, subgrid_size):
    """Determine the order of numbers for a cell based on the Least Constraining Value heuristic."""
    lcv_scores = {}
    for num in range(1, grid_size + 1):
        if is_valid(board, row, col, num, grid_size, subgrid_size):
            # Temporarily assign the number
            board[row][col] = num
            # Count the number of valid values left for all unassigned cells
            score = 0
            for r in range(grid_size):
                for c in range(grid_size):
                    if board[r][c] == 0:
                        domain = [
                            n for n in range(1, grid_size + 1)
                            if is_valid(board, r, c, n, grid_size, subgrid_size)
                        ]
                        score += len(domain)
            lcv_scores[num] = score
            board[row][col] = 0  # Undo the temporary assignment

    # Sort numbers by their scores in descending order (least constraining value first)
    return sorted(lcv_scores, key=lcv_scores.get)


def generate_full_grid(grid_size, subgrid_size):
    """Generate a fully solved Sudoku grid."""
    board = [[0 for _ in range(grid_size)] for _ in range(grid_size)]
    solve_sudoku(board, grid_size, subgrid_size)
    return board


def remove_numbers_from_grid(board, empty_cells):
    """Remove numbers from a solved grid to create a puzzle with a given number of empty cells."""
    grid_size = len(board)
    cells = [(row, col) for row in range(grid_size) for col in range(grid_size)]
    random.shuffle(cells)
    
    for _ in range(empty_cells):
        if not cells:
            break
        row, col = cells.pop()
        board[row][col] = 0


def solve_sudoku(board, grid_size, subgrid_size, heuristic="mrv"):
    """Solve the Sudoku puzzle using specified heuristic."""
    if heuristic == "mrv":
        mrv_cell = find_mrv_cell(board, grid_size, subgrid_size)
        if not mrv_cell:
            return True  # Puzzle solved
        row, col = mrv_cell

        for num in range(1, grid_size + 1):
            if is_valid(board, row, col, num, grid_size, subgrid_size):
                board[row][col] = num
                if solve_sudoku(board, grid_size, subgrid_size, heuristic):  # Recurse
                    return True
                board[row][col] = 0  # Undo placement (backtrack)
        return False

    elif heuristic == "lcv":
        mrv_cell = find_mrv_cell(board, grid_size, subgrid_size)
        if not mrv_cell:
            return True
        row, col = mrv_cell
        lcv_order = get_lcv_order(board, row, col, grid_size, subgrid_size)
        for num in lcv_order:
            if is_valid(board, row, col, num, grid_size, subgrid_size):
                board[row][col] = num
                if solve_sudoku(board, grid_size, subgrid_size, heuristic):
                    return True
                board[row][col] = 0
        return False

    return False  # Default fail-safe


def print_board_framed(board):
    """Print the Sudoku board in a readable frame format."""
    grid_size = len(board)
    subgrid_size = int(math.sqrt(grid_size))
    horizontal_line = "+".join(["-" * (subgrid_size * 2 + 1)] * subgrid_size)
    
    for i, row in enumerate(board):
        if i % subgrid_size == 0:
            print(horizontal_line)
        row_text = " | ".join(
            " ".join(str(cell) if cell != 0 else "." for cell in row[j:j + subgrid_size])
            for j in range(0, grid_size, subgrid_size)
        )
        print(row_text)
    print(horizontal_line)


def solve_sudoku_combined(board, grid_size, subgrid_size):
    """Solve the Sudoku puzzle using MRV for variable selection and LCV for value selection."""
    # Find the most constrained variable (MRV)
    mrv_cell = find_mrv_cell(board, grid_size, subgrid_size)
    if not mrv_cell:
        return True  # Puzzle solved

    row, col = mrv_cell

    # Determine the least constraining value order (LCV)
    lcv_order = get_lcv_order(board, row, col, grid_size, subgrid_size)
    for num in lcv_order:
        if is_valid(board, row, col, num, grid_size, subgrid_size):
            board[row][col] = num
            if solve_sudoku_combined(board, grid_size, subgrid_size):  # Recurse
                return True
            board[row][col] = 0  # Undo placement (backtrack)

    return False  # No solution found




# Use this hybrid solver in your main sudoku function
def sudoku(grid_size, difficulty):
    """Generate and solve a Sudoku puzzle with hybrid solving techniques."""
    if int(math.sqrt(grid_size)) ** 2 != grid_size:
        raise ValueError("Grid size must be a perfect square (e.g., 9x9, 16x16).")
    subgrid_size = int(math.sqrt(grid_size))

    # Define difficulty levels
    difficulty_levels = {
        "easy": grid_size ** 2 // 3,
        "medium": grid_size ** 2 // 2,
        "hard": grid_size ** 2 * 2 // 3
    }
    if difficulty not in difficulty_levels:
        raise ValueError("Invalid difficulty level. Choose 'easy', 'medium', or 'hard'.")

    # Generate a full grid and remove numbers
    empty_cells = difficulty_levels[difficulty]
    sudoku_grid = generate_full_grid(grid_size, subgrid_size)
    remove_numbers_from_grid(sudoku_grid, empty_cells)

    print(f"Generated {difficulty} Sudoku grid ({grid_size}x{grid_size}):")
    print_board_framed(sudoku_grid)
    print("\nSolving the Sudoku grid...")

    # Solve with hybrid method
    if hybrid_solve_sudoku(sudoku_grid, grid_size, subgrid_size):
        print("\nSolved Sudoku grid:")
        print_board_framed(sudoku_grid)
    else:
        print("No solution found.")

In [2]:
def forward_checking(board, grid_size, subgrid_size):
    """Update domains after assigning a value to ensure constraints hold."""
    for row in range(grid_size):
        for col in range(grid_size):
            if board[row][col] == 0:
                domain = [num for num in range(1, grid_size + 1) if is_valid(board, row, col, num, grid_size, subgrid_size)]
                if not domain:
                    return False  # If any variable's domain becomes empty, backtrack
    return True

def solve_sudoku_with_forward_checking(board, grid_size, subgrid_size):
    """Solve Sudoku using forward checking."""
    mrv_cell = find_mrv_cell(board, grid_size, subgrid_size)
    if not mrv_cell:
        return True  # Puzzle solved
    
    row, col = mrv_cell
    domain = [num for num in range(1, grid_size + 1) if is_valid(board, row, col, num, grid_size, subgrid_size)]
    
    for num in domain:
        board[row][col] = num
        if forward_checking(board, grid_size, subgrid_size):
            if solve_sudoku_with_forward_checking(board, grid_size, subgrid_size):
                return True
        board[row][col] = 0  # Backtrack
    
    return False

In [3]:
def find_degree_cell(board, grid_size, subgrid_size):
    """Find the cell with the most constraints (Degree Heuristic)."""
    max_degree = -1
    degree_cell = None
    
    for row in range(grid_size):
        for col in range(grid_size):
            if board[row][col] == 0:  # Only consider empty cells
                constraints = sum(1 for i in range(grid_size) if board[row][i] == 0 or board[i][col] == 0)
                subgrid_row = subgrid_size * (row // subgrid_size)
                subgrid_col = subgrid_size * (col // subgrid_size)
                constraints += sum(
                    1 for i in range(subgrid_size) for j in range(subgrid_size)
                    if board[subgrid_row + i][subgrid_col + j] == 0
                )
                if constraints > max_degree:
                    max_degree = constraints
                    degree_cell = (row, col)
    return degree_cell

def solve_sudoku_with_degree(board, grid_size, subgrid_size):
    """Solve Sudoku using Degree Heuristic."""
    degree_cell = find_degree_cell(board, grid_size, subgrid_size)
    if not degree_cell:
        return True  # Puzzle solved
    
    row, col = degree_cell
    domain = [num for num in range(1, grid_size + 1) if is_valid(board, row, col, num, grid_size, subgrid_size)]
    
    for num in domain:
        board[row][col] = num
        if solve_sudoku_with_degree(board, grid_size, subgrid_size):
            return True
        board[row][col] = 0  # Backtrack
    
    return False

In [4]:
from collections import deque

def ac3(board, grid_size, subgrid_size):
    """Enforce arc consistency using the AC-3 algorithm."""
    def get_neighbors(row, col):
        """Get all cells that share a row, column, or subgrid with (row, col)."""
        neighbors = set()
        for i in range(grid_size):
            if i != col:
                neighbors.add((row, i))
            if i != row:
                neighbors.add((i, col))
        subgrid_row_start = (row // subgrid_size) * subgrid_size
        subgrid_col_start = (col // subgrid_size) * subgrid_size
        for r in range(subgrid_row_start, subgrid_row_start + subgrid_size):
            for c in range(subgrid_col_start, subgrid_col_start + subgrid_size):
                if (r, c) != (row, col):
                    neighbors.add((r, c))
        return neighbors

    def remove_inconsistent_values(row1, col1, row2, col2):
        """Remove inconsistent values between two variables."""
        removed = False
        if board[row2][col2] != 0:
            if board[row1][col1] == 0:
                domain = [num for num in range(1, grid_size + 1) if is_valid(board, row1, col1, num, grid_size, subgrid_size)]
                if board[row2][col2] in domain:
                    domain.remove(board[row2][col2])
                    if len(domain) == 1:
                        board[row1][col1] = domain[0]
                        removed = True
        return removed

    # Initialize the queue with all arcs
    queue = deque()
    for row in range(grid_size):
        for col in range(grid_size):
            if board[row][col] == 0:
                neighbors = get_neighbors(row, col)
                for neighbor in neighbors:
                    queue.append(((row, col), neighbor))

    # Process the queue
    while queue:
        (row1, col1), (row2, col2) = queue.popleft()
        if remove_inconsistent_values(row1, col1, row2, col2):
            neighbors = get_neighbors(row1, col1)
            for neighbor in neighbors:
                if neighbor != (row2, col2):
                    queue.append((neighbor, (row1, col1)))

    # Check if the board is still valid
    for row in range(grid_size):
        for col in range(grid_size):
            if board[row][col] == 0:
                if not any(is_valid(board, row, col, num, grid_size, subgrid_size) for num in range(1, grid_size + 1)):
                    return False
    return True

def solve_sudoku_with_ac3(board, grid_size, subgrid_size):
    """Solve Sudoku using AC-3 for preprocessing followed by backtracking."""
    # Apply AC-3 preprocessing
    if not ac3(board, grid_size, subgrid_size):
        return False  # Puzzle is unsolvable after AC-3

    # Use backtracking to complete the solution if AC-3 didn't fully solve it
    return solve_sudoku(board, grid_size, subgrid_size)

In [5]:
def measure_sudoku_performance_with_heuristics():
    """Measure the performance of various heuristics on a 9x9 Sudoku grid."""
    grid_size = 4
    heuristics = {
        "MRV": solve_sudoku,
        "Degree": solve_sudoku_with_degree,
        "Forward Checking": solve_sudoku_with_forward_checking,
        "AC-3": solve_sudoku_with_ac3,
        "Combined (MRV + LCV)": solve_sudoku_combined
    }
    difficulties = ["easy", "medium", "hard"]
    results = []

    for heuristic_name, solver in heuristics.items():
        for difficulty in difficulties:
            sudoku_grid = generate_full_grid(grid_size, int(math.sqrt(grid_size)))
            remove_numbers_from_grid(
                sudoku_grid,
                {"easy": grid_size**2 // 3, "medium": grid_size**2 // 2, "hard": 2 * grid_size**2 // 3}[difficulty]
            )
            start_time = time.time()
            solver(sudoku_grid, grid_size, int(math.sqrt(grid_size)))
            end_time = time.time()
            results.append({
                "Grid Size": grid_size,
                "Heuristic": heuristic_name,
                "Difficulty": difficulty,
                "Time Taken (s)": round(end_time - start_time, 4)
            })

    return pd.DataFrame(results)

# Measure performance


In [6]:
def measure_sudoku_performance_with_heuristics(difficulties, heuristics, grid_size=9):
    """
    Measure the performance of various heuristics on a Sudoku grid.
    Also returns the initial grid, solved grid, and name of the algorithm.
    """
    results = []
    detailed_results = []  # To store initial grid, solved grid, and other details

    for heuristic_name, solver in heuristics.items():
        for difficulty in difficulties:
            # Generate initial grid and remove numbers based on difficulty
            initial_grid = generate_full_grid(grid_size, int(math.sqrt(grid_size)))
            remove_numbers_from_grid(
                initial_grid,
                {"easy": grid_size**2 // 3, "medium": grid_size**2 // 2, "hard": 2 * grid_size**2 // 3}[difficulty]
            )
            
            # Copy the grid for solving
            sudoku_grid = [row[:] for row in initial_grid]

            start_time = time.time()
            solved = solver(sudoku_grid, grid_size, int(math.sqrt(grid_size)))
            end_time = time.time()

            # Append performance results
            results.append({
                "Grid Size": grid_size,
                "Heuristic": heuristic_name,
                "Difficulty": difficulty,
                "Time Taken (s)": round(end_time - start_time, 4),
                "Solved": solved  # True if the puzzle is solved, False otherwise
            })

            # Append detailed results
            detailed_results.append({
                "Heuristic": heuristic_name,
                "Difficulty": difficulty,
                "Initial Grid": initial_grid,
                "Solved Grid": sudoku_grid if solved else None,
                "Time Taken (s)": round(end_time - start_time, 4)
            })

    # Create DataFrame for performance
    performance_df = pd.DataFrame(results).sort_values(by="Time Taken (s)", ascending=True)

    return performance_df, detailed_results

In [7]:
    heuristics = {
        "MRV": solve_sudoku,
        "Degree": solve_sudoku_with_degree,
        "Forward Checking": solve_sudoku_with_forward_checking,
        "AC-3": solve_sudoku_with_ac3,
        #"Combined (MRV + LCV)": solve_sudoku_combined,
    }

performance_df = measure_sudoku_performance_with_heuristics(['easy'],heuristics)
performance_df

(   Grid Size         Heuristic Difficulty  Time Taken (s)  Solved
 0          9               MRV       easy          0.0054    True
 3          9              AC-3       easy          0.0093    True
 2          9  Forward Checking       easy          0.0096    True
 1          9            Degree       easy          0.0317    True,
 [{'Heuristic': 'MRV',
   'Difficulty': 'easy',
   'Initial Grid': [[1, 2, 0, 4, 0, 0, 7, 8, 9],
    [4, 0, 6, 7, 8, 0, 1, 0, 3],
    [7, 8, 0, 1, 2, 0, 4, 5, 6],
    [2, 3, 0, 6, 0, 0, 8, 9, 5],
    [8, 7, 0, 0, 0, 2, 3, 0, 0],
    [6, 9, 4, 5, 3, 8, 2, 1, 7],
    [3, 1, 0, 2, 0, 0, 0, 4, 8],
    [0, 4, 2, 8, 9, 7, 0, 3, 0],
    [9, 0, 0, 3, 0, 0, 5, 7, 2]],
   'Solved Grid': [[1, 2, 3, 4, 5, 6, 7, 8, 9],
    [4, 5, 6, 7, 8, 9, 1, 2, 3],
    [7, 8, 9, 1, 2, 3, 4, 5, 6],
    [2, 3, 1, 6, 7, 4, 8, 9, 5],
    [8, 7, 5, 9, 1, 2, 3, 6, 4],
    [6, 9, 4, 5, 3, 8, 2, 1, 7],
    [3, 1, 7, 2, 6, 5, 9, 4, 8],
    [5, 4, 2, 8, 9, 7, 6, 3, 1],
    [9, 6, 8, 3, 4, 1, 

In [8]:
    heuristics = {
        "MRV": solve_sudoku,
        "Degree": solve_sudoku_with_degree,
        "Forward Checking": solve_sudoku_with_forward_checking,
        "AC-3": solve_sudoku_with_ac3,
        #"Combined (MRV + LCV)": solve_sudoku_combined,
    }

performance_df = measure_sudoku_performance_with_heuristics(['medium'],heuristics)
performance_df

(   Grid Size         Heuristic Difficulty  Time Taken (s)  Solved
 0          9               MRV     medium          0.0112    True
 3          9              AC-3     medium          0.0182    True
 2          9  Forward Checking     medium          0.0223    True
 1          9            Degree     medium          1.5712    True,
 [{'Heuristic': 'MRV',
   'Difficulty': 'medium',
   'Initial Grid': [[0, 2, 3, 0, 5, 6, 0, 0, 9],
    [0, 5, 0, 0, 8, 9, 1, 2, 3],
    [7, 0, 0, 0, 0, 3, 0, 0, 6],
    [2, 3, 0, 0, 7, 0, 0, 0, 0],
    [8, 7, 0, 0, 1, 0, 0, 0, 0],
    [6, 0, 4, 5, 0, 0, 2, 1, 0],
    [0, 1, 7, 2, 6, 0, 0, 0, 0],
    [5, 0, 0, 8, 9, 7, 6, 3, 0],
    [0, 6, 8, 3, 4, 1, 5, 0, 0]],
   'Solved Grid': [[1, 2, 3, 4, 5, 6, 7, 8, 9],
    [4, 5, 6, 7, 8, 9, 1, 2, 3],
    [7, 8, 9, 1, 2, 3, 4, 5, 6],
    [2, 3, 1, 6, 7, 4, 8, 9, 5],
    [8, 7, 5, 9, 1, 2, 3, 6, 4],
    [6, 9, 4, 5, 3, 8, 2, 1, 7],
    [3, 1, 7, 2, 6, 5, 9, 4, 8],
    [5, 4, 2, 8, 9, 7, 6, 3, 1],
    [9, 6, 8, 3, 4, 1

In [9]:
heuristics = {
        "MRV": solve_sudoku,
        #"Degree": solve_sudoku_with_degree,
        "Forward Checking": solve_sudoku_with_forward_checking,
        "AC-3": solve_sudoku_with_ac3,
        "Combined (MRV + LCV)": solve_sudoku_combined,
    }

performance_df = measure_sudoku_performance_with_heuristics(['easy'],heuristics)
performance_df

(   Grid Size             Heuristic Difficulty  Time Taken (s)  Solved
 0          9                   MRV       easy          0.0048    True
 1          9      Forward Checking       easy          0.0090    True
 3          9  Combined (MRV + LCV)       easy          0.0095    True
 2          9                  AC-3       easy          0.0104    True,
 [{'Heuristic': 'MRV',
   'Difficulty': 'easy',
   'Initial Grid': [[1, 0, 3, 0, 5, 0, 0, 8, 9],
    [0, 5, 0, 7, 0, 0, 1, 2, 3],
    [7, 8, 0, 1, 2, 3, 4, 5, 0],
    [2, 0, 1, 6, 7, 0, 8, 9, 5],
    [0, 7, 5, 9, 1, 0, 0, 6, 0],
    [6, 9, 0, 5, 3, 8, 0, 1, 7],
    [3, 1, 0, 2, 6, 5, 9, 4, 8],
    [0, 0, 2, 0, 9, 7, 6, 0, 0],
    [9, 6, 0, 3, 4, 1, 5, 0, 0]],
   'Solved Grid': [[1, 2, 3, 4, 5, 6, 7, 8, 9],
    [4, 5, 6, 7, 8, 9, 1, 2, 3],
    [7, 8, 9, 1, 2, 3, 4, 5, 6],
    [2, 3, 1, 6, 7, 4, 8, 9, 5],
    [8, 7, 5, 9, 1, 2, 3, 6, 4],
    [6, 9, 4, 5, 3, 8, 2, 1, 7],
    [3, 1, 7, 2, 6, 5, 9, 4, 8],
    [5, 4, 2, 8, 9, 7, 6, 3, 1],
   

In [10]:
heuristics = {
        "MRV": solve_sudoku,
        #"Degree": solve_sudoku_with_degree,
        "Forward Checking": solve_sudoku_with_forward_checking,
        "AC-3": solve_sudoku_with_ac3,
        "Combined (MRV + LCV)": solve_sudoku_combined,
    }


performance_df = measure_sudoku_performance_with_heuristics(['medium'],heuristics)
performance_df

(   Grid Size             Heuristic Difficulty  Time Taken (s)  Solved
 0          9                   MRV     medium          0.0117    True
 2          9                  AC-3     medium          0.0182    True
 1          9      Forward Checking     medium          0.0241    True
 3          9  Combined (MRV + LCV)     medium          0.0241    True,
 [{'Heuristic': 'MRV',
   'Difficulty': 'medium',
   'Initial Grid': [[1, 2, 3, 4, 5, 6, 0, 8, 9],
    [0, 5, 6, 7, 0, 9, 1, 0, 0],
    [7, 0, 9, 0, 2, 0, 4, 0, 0],
    [0, 3, 1, 0, 0, 0, 8, 9, 0],
    [0, 7, 5, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 5, 0, 8, 2, 0, 0],
    [3, 0, 0, 2, 6, 0, 0, 0, 8],
    [0, 0, 2, 8, 9, 0, 0, 0, 1],
    [9, 0, 0, 3, 0, 1, 5, 7, 2]],
   'Solved Grid': [[1, 2, 3, 4, 5, 6, 7, 8, 9],
    [4, 5, 6, 7, 8, 9, 1, 2, 3],
    [7, 8, 9, 1, 2, 3, 4, 5, 6],
    [2, 3, 1, 6, 7, 4, 8, 9, 5],
    [8, 7, 5, 9, 1, 2, 3, 6, 4],
    [6, 9, 4, 5, 3, 8, 2, 1, 7],
    [3, 1, 7, 2, 6, 5, 9, 4, 8],
    [5, 4, 2, 8, 9, 7, 6, 3, 1],
 

In [11]:
heuristics = {
        "MRV": solve_sudoku,
        #"Degree": solve_sudoku_with_degree,
        "Forward Checking": solve_sudoku_with_forward_checking,
        "AC-3": solve_sudoku_with_ac3,
        "Combined (MRV + LCV)": solve_sudoku_combined,
    }


performance_df = measure_sudoku_performance_with_heuristics(['hard'],heuristics)
performance_df

(   Grid Size             Heuristic Difficulty  Time Taken (s)  Solved
 2          9                  AC-3       hard          0.0438    True
 1          9      Forward Checking       hard          0.0514    True
 3          9  Combined (MRV + LCV)       hard          0.1817    True
 0          9                   MRV       hard          0.3351    True,
 [{'Heuristic': 'MRV',
   'Difficulty': 'hard',
   'Initial Grid': [[0, 0, 3, 0, 0, 6, 0, 8, 9],
    [0, 5, 0, 0, 8, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 3, 0, 0, 0],
    [2, 0, 0, 0, 7, 0, 8, 9, 0],
    [0, 7, 0, 9, 0, 0, 0, 0, 0],
    [0, 9, 0, 0, 0, 0, 2, 1, 7],
    [3, 0, 7, 2, 0, 5, 0, 0, 0],
    [5, 0, 0, 0, 0, 0, 0, 3, 0],
    [0, 0, 8, 3, 0, 0, 0, 7, 2]],
   'Solved Grid': [[4, 1, 3, 5, 2, 6, 7, 8, 9],
    [9, 5, 6, 1, 8, 7, 3, 2, 4],
    [7, 8, 2, 4, 9, 3, 1, 5, 6],
    [2, 3, 4, 6, 7, 1, 8, 9, 5],
    [8, 7, 1, 9, 5, 2, 4, 6, 3],
    [6, 9, 5, 8, 3, 4, 2, 1, 7],
    [3, 6, 7, 2, 1, 5, 9, 4, 8],
    [5, 2, 9, 7, 4, 8, 6, 3, 1],
   

In [12]:
def hybrid_solve_sudoku(board, grid_size, subgrid_size):
    """Solve Sudoku using AC-3 preprocessing with MRV and LCV heuristics."""
    # Preprocess with AC-3
    if not ac3(board, grid_size, subgrid_size):
        return False  # Puzzle is unsolvable
    
    # Apply MRV + LCV backtracking
    mrv_cell = find_mrv_cell(board, grid_size, subgrid_size)
    if not mrv_cell:
        return True  # Puzzle solved

    row, col = mrv_cell
    lcv_order = get_lcv_order(board, row, col, grid_size, subgrid_size)
    for num in lcv_order:
        if is_valid(board, row, col, num, grid_size, subgrid_size):
            board[row][col] = num
            if hybrid_solve_sudoku(board, grid_size, subgrid_size):
                return True
            board[row][col] = 0  # Backtrack
    return False  # No solution found

In [13]:
heuristics = {
        #"MRV": solve_sudoku,
        #"Degree": solve_sudoku_with_degree,
        #"Forward Checking": solve_sudoku_with_forward_checking,
        "AC-3": solve_sudoku_with_ac3,
        #"Combined (MRV + LCV)": solve_sudoku_combined,
        "AC3+MRV+LCV": hybrid_solve_sudoku
    }

performance_df = measure_sudoku_performance_with_heuristics(['hard'],heuristics,grid_size=9)
performance_df

(   Grid Size    Heuristic Difficulty  Time Taken (s)  Solved
 0          9         AC-3       hard          0.0356    True
 1          9  AC3+MRV+LCV       hard          7.2434    True,
 [{'Heuristic': 'AC-3',
   'Difficulty': 'hard',
   'Initial Grid': [[0, 2, 3, 0, 5, 0, 7, 0, 0],
    [0, 0, 0, 7, 0, 0, 0, 0, 0],
    [0, 0, 9, 1, 2, 3, 0, 0, 0],
    [0, 3, 0, 0, 0, 4, 0, 0, 5],
    [8, 0, 0, 0, 0, 0, 3, 6, 4],
    [0, 0, 4, 0, 3, 0, 0, 1, 0],
    [0, 0, 0, 0, 6, 0, 0, 0, 0],
    [0, 0, 0, 8, 9, 7, 0, 3, 0],
    [0, 0, 0, 0, 0, 1, 5, 7, 0]],
   'Solved Grid': [[1, 2, 3, 4, 5, 6, 7, 8, 9],
    [4, 6, 5, 7, 8, 9, 1, 2, 3],
    [7, 8, 9, 1, 2, 3, 4, 5, 6],
    [6, 3, 7, 2, 1, 4, 8, 9, 5],
    [8, 1, 2, 9, 7, 5, 3, 6, 4],
    [9, 5, 4, 6, 3, 8, 2, 1, 7],
    [3, 7, 8, 5, 6, 2, 9, 4, 1],
    [5, 4, 1, 8, 9, 7, 6, 3, 2],
    [2, 9, 6, 3, 4, 1, 5, 7, 8]],
   'Time Taken (s)': 0.0356},
  {'Heuristic': 'AC3+MRV+LCV',
   'Difficulty': 'hard',
   'Initial Grid': [[0, 0, 0, 4, 5, 6, 0, 8, 0],
  

In [14]:
heuristics = {
        #"MRV": solve_sudoku,
        #"Degree": solve_sudoku_with_degree,
        #"Forward Checking": solve_sudoku_with_forward_checking,
        "AC-3": solve_sudoku_with_ac3,
        #"Combined (MRV + LCV)": solve_sudoku_combined,
        "AC3+MRV+LCV": hybrid_solve_sudoku
    }

performance_df = measure_sudoku_performance_with_heuristics(['medium'],heuristics,grid_size=9)
performance_df

(   Grid Size    Heuristic Difficulty  Time Taken (s)  Solved
 0          9         AC-3     medium          0.0197    True
 1          9  AC3+MRV+LCV     medium          0.1924    True,
 [{'Heuristic': 'AC-3',
   'Difficulty': 'medium',
   'Initial Grid': [[0, 0, 0, 4, 0, 6, 0, 8, 9],
    [4, 5, 6, 0, 8, 0, 0, 2, 0],
    [0, 8, 0, 0, 0, 0, 0, 5, 6],
    [2, 3, 0, 0, 7, 0, 8, 0, 5],
    [0, 7, 5, 0, 0, 2, 0, 0, 0],
    [0, 0, 0, 5, 3, 8, 2, 0, 0],
    [0, 0, 7, 0, 6, 5, 9, 4, 0],
    [5, 4, 2, 8, 0, 7, 0, 3, 1],
    [9, 6, 0, 3, 0, 1, 5, 0, 0]],
   'Solved Grid': [[1, 2, 3, 4, 5, 6, 7, 8, 9],
    [4, 5, 6, 7, 8, 9, 1, 2, 3],
    [7, 8, 9, 1, 2, 3, 4, 5, 6],
    [2, 3, 1, 6, 7, 4, 8, 9, 5],
    [8, 7, 5, 9, 1, 2, 3, 6, 4],
    [6, 9, 4, 5, 3, 8, 2, 1, 7],
    [3, 1, 7, 2, 6, 5, 9, 4, 8],
    [5, 4, 2, 8, 9, 7, 6, 3, 1],
    [9, 6, 8, 3, 4, 1, 5, 7, 2]],
   'Time Taken (s)': 0.0197},
  {'Heuristic': 'AC3+MRV+LCV',
   'Difficulty': 'medium',
   'Initial Grid': [[0, 0, 0, 0, 0, 6, 0, 8, 0]

In [15]:
heuristics = {
        #"MRV": solve_sudoku,
        #"Degree": solve_sudoku_with_degree,
        #"Forward Checking": solve_sudoku_with_forward_checking,
        "AC-3": solve_sudoku_with_ac3,
        #"Combined (MRV + LCV)": solve_sudoku_combined,
        "AC3+MRV+LCV": hybrid_solve_sudoku
    }

performance_df = measure_sudoku_performance_with_heuristics(['easy'],heuristics,grid_size=9)
performance_df

(   Grid Size    Heuristic Difficulty  Time Taken (s)  Solved
 0          9         AC-3       easy          0.0105    True
 1          9  AC3+MRV+LCV       easy          0.0834    True,
 [{'Heuristic': 'AC-3',
   'Difficulty': 'easy',
   'Initial Grid': [[1, 2, 3, 0, 5, 6, 0, 0, 0],
    [4, 0, 6, 7, 8, 9, 0, 2, 3],
    [7, 8, 9, 0, 2, 0, 0, 5, 0],
    [0, 0, 1, 0, 7, 4, 8, 9, 5],
    [8, 7, 5, 9, 1, 2, 0, 0, 0],
    [6, 9, 4, 5, 3, 0, 2, 0, 7],
    [0, 1, 7, 2, 6, 5, 9, 4, 0],
    [5, 4, 0, 8, 0, 0, 6, 3, 1],
    [9, 6, 0, 0, 0, 0, 5, 7, 2]],
   'Solved Grid': [[1, 2, 3, 4, 5, 6, 7, 8, 9],
    [4, 5, 6, 7, 8, 9, 1, 2, 3],
    [7, 8, 9, 1, 2, 3, 4, 5, 6],
    [2, 3, 1, 6, 7, 4, 8, 9, 5],
    [8, 7, 5, 9, 1, 2, 3, 6, 4],
    [6, 9, 4, 5, 3, 8, 2, 1, 7],
    [3, 1, 7, 2, 6, 5, 9, 4, 8],
    [5, 4, 2, 8, 9, 7, 6, 3, 1],
    [9, 6, 8, 3, 4, 1, 5, 7, 2]],
   'Time Taken (s)': 0.0105},
  {'Heuristic': 'AC3+MRV+LCV',
   'Difficulty': 'easy',
   'Initial Grid': [[1, 0, 3, 0, 5, 0, 7, 0, 9],
  

In [16]:
initial_grid = [
    [9, 1, 3, 0, 0, 0, 5, 0, 0],
    [6, 0, 7, 0, 0, 0, 0, 2, 4],
    [0, 5, 0, 0, 8, 0, 0, 7, 0],
    [0, 7, 9, 0, 0, 0, 0, 0, 0],
    [0, 0, 2, 0, 9, 0, 0, 4, 3],
    [0, 0, 0, 0, 0, 4, 0, 9, 0],
    [0, 4, 0, 0, 0, 1, 9, 0, 0],
    [7, 0, 6, 0, 0, 9, 0, 0, 5],
    [0, 0, 1, 0, 0, 6, 4, 0, 7]
]

# Define grid size and subgrid size
grid_size = 9
subgrid_size = 3

# Solve the Sudoku
solved_grid = [row[:] for row in initial_grid]  # Copy of the initial grid
solve_sudoku(solved_grid, grid_size, subgrid_size)

# Display the solved Sudoku grid
solved_grid

[[9, 1, 3, 4, 2, 7, 5, 8, 6],
 [6, 8, 7, 9, 1, 5, 3, 2, 4],
 [2, 5, 4, 6, 8, 3, 1, 7, 9],
 [4, 7, 9, 1, 3, 2, 6, 5, 8],
 [1, 6, 2, 5, 9, 8, 7, 4, 3],
 [5, 3, 8, 7, 6, 4, 2, 9, 1],
 [3, 4, 5, 8, 7, 1, 9, 6, 2],
 [7, 2, 6, 3, 4, 9, 8, 1, 5],
 [8, 9, 1, 2, 5, 6, 4, 3, 7]]

In [17]:
initial_grid = [
    [9, 1, 3, 0, 0, 0, 5, 0, 0],
    [6, 0, 7, 0, 0, 0, 0, 2, 4],
    [0, 5, 0, 0, 8, 0, 0, 7, 0],
    [0, 7, 9, 0, 0, 0, 0, 0, 0],
    [0, 0, 2, 0, 9, 0, 0, 4, 3],
    [0, 0, 0, 0, 0, 4, 0, 9, 0],
    [0, 4, 0, 0, 0, 1, 9, 0, 0],
    [7, 0, 6, 0, 0, 9, 0, 0, 5],
    [0, 0, 1, 0, 0, 6, 4, 0, 7]
]

# Define grid size and subgrid size
grid_size = 9
subgrid_size = 3

# Solve the Sudoku
solved_grid = [row[:] for row in initial_grid]  # Copy of the initial grid
solve_sudoku(solved_grid, grid_size, subgrid_size, "lcv")

# Display the solved Sudoku grid
solved_grid

[[9, 1, 3, 4, 2, 7, 5, 8, 6],
 [6, 8, 7, 9, 1, 5, 3, 2, 4],
 [2, 5, 4, 6, 8, 3, 1, 7, 9],
 [4, 7, 9, 1, 3, 2, 6, 5, 8],
 [1, 6, 2, 5, 9, 8, 7, 4, 3],
 [5, 3, 8, 7, 6, 4, 2, 9, 1],
 [3, 4, 5, 8, 7, 1, 9, 6, 2],
 [7, 2, 6, 3, 4, 9, 8, 1, 5],
 [8, 9, 1, 2, 5, 6, 4, 3, 7]]

In [18]:
heuristics = {
    #"mrv": solve_sudoku,
    #"lcv": lambda b, g, s: solve_sudoku(b, g, s, "lcv"),
    #"combined": solve_sudoku_combined,
    "ac3": solve_sudoku_with_ac3
}

# Measure performance
performance_df, detailed_results = measure_sudoku_performance_with_heuristics(["easy", "medium"], heuristics)

# Display the performance DataFrame
print(performance_df)

# Access detailed results
for result in detailed_results:
    print("Heuristic:", result["Heuristic"])
    print("Difficulty:", result["Difficulty"])
    print("Initial Grid:")
    print_board_framed(result["Initial Grid"])
    if result["Solved Grid"]:
        print("Solved Grid:")
        print_board_framed(result["Solved Grid"])
    else:
        print("No solution found.")
    print("Time Taken (s):", result["Time Taken (s)"])
    print("-" * 30)

   Grid Size Heuristic Difficulty  Time Taken (s)  Solved
0          9       ac3       easy          0.0100    True
1          9       ac3     medium          0.0184    True
Heuristic: ac3
Difficulty: easy
Initial Grid:
-------+-------+-------
1 2 3 | 4 5 . | 7 8 9
4 5 . | 7 8 9 | 1 2 3
7 8 . | 1 2 3 | 4 5 6
-------+-------+-------
. . 1 | 6 . 4 | . . 5
. . 5 | 9 . . | 3 . .
. 9 4 | 5 3 8 | . . .
-------+-------+-------
3 . . | 2 6 5 | . 4 8
5 . . | 8 9 7 | 6 . .
9 6 8 | 3 . . | 5 7 2
-------+-------+-------
Solved Grid:
-------+-------+-------
1 2 3 | 4 5 6 | 7 8 9
4 5 6 | 7 8 9 | 1 2 3
7 8 9 | 1 2 3 | 4 5 6
-------+-------+-------
2 3 1 | 6 7 4 | 8 9 5
8 7 5 | 9 1 2 | 3 6 4
6 9 4 | 5 3 8 | 2 1 7
-------+-------+-------
3 1 7 | 2 6 5 | 9 4 8
5 4 2 | 8 9 7 | 6 3 1
9 6 8 | 3 4 1 | 5 7 2
-------+-------+-------
Time Taken (s): 0.01
------------------------------
Heuristic: ac3
Difficulty: medium
Initial Grid:
-------+-------+-------
1 . 3 | . 5 6 | 7 . .
. . . | . 8 9 | . 2 .
7 . 9 | 1 2

In [19]:
def step_measure_sudoku_performance_with_heuristics(difficulties, heuristics, grid_size=9):
    """
    Measure the performance of various heuristics on a Sudoku grid.
    Returns performance data including initial grid, intermediate steps (first 5 Sudoku states), and the solved grid.
    """
    results = []
    detailed_results = []  # To store detailed information

    def solver_with_steps(solver, board, grid_size, subgrid_size):
        """Wrapper to capture first 5 steps and their resulting Sudoku grids."""
        steps = []
        snapshots = []  # To store the board state after each step
        total_steps = 0

        def track_step(row, col, value):
            nonlocal total_steps
            total_steps += 1
            if len(steps) < 5:
                steps.append((row, col, value))
                # Capture the current state of the board
                snapshots.append([row[:] for row in board])

        # Modify the solver to track steps
        original_is_valid = is_valid

        def wrapped_is_valid(board, row, col, num, grid_size, subgrid_size):
            if board[row][col] == 0 and num != 0:
                track_step(row, col, num)
            return original_is_valid(board, row, col, num, grid_size, subgrid_size)
        
        globals()['is_valid'] = wrapped_is_valid
        solved = solver(board, grid_size, subgrid_size)
        globals()['is_valid'] = original_is_valid  # Restore original function

        return solved, steps, snapshots, total_steps

    for heuristic_name, solver in heuristics.items():
        for difficulty in difficulties:
            # Generate initial grid and remove numbers based on difficulty
            initial_grid = generate_full_grid(grid_size, int(math.sqrt(grid_size)))
            remove_numbers_from_grid(
                initial_grid,
                {"easy": random.randint(32, 36), "medium": random.randint(37, 41), "hard": random.randint(42, 49)}[difficulty]
            )
            
            # Copy the grid for solving
            sudoku_grid = [row[:] for row in initial_grid]

            start_time = time.time()
            solved, steps, snapshots, total_steps = solver_with_steps(solver, sudoku_grid, grid_size, int(math.sqrt(grid_size)))
            end_time = time.time()

            # Append performance results
            results.append({
                "Heuristic": heuristic_name,
                "Difficulty": difficulty,
                "Time Taken (s)": round(end_time - start_time, 4),
                "Solved": solved,
                "Total Steps": total_steps
            })

            # Append detailed results
            detailed_results.append({
                "Heuristic": heuristic_name,
                "Difficulty": difficulty,
                "Initial Grid": initial_grid,
                "First 5 Steps": snapshots,  # Store the board state after each step
                "Solved Grid": sudoku_grid if solved else None,
                "Total Steps": total_steps,
                "Time Taken (s)": round(end_time - start_time, 4)
            })

    # Create DataFrame for performance
    performance_df = pd.DataFrame(results).sort_values(by="Time Taken (s)", ascending=True)

    return performance_df, detailed_results


##############
##############
############## USAGE
heuristics = {
    "mrv": solve_sudoku,
    "lcv": lambda b, g, s: solve_sudoku(b, g, s, "lcv"),
    "combined": solve_sudoku_combined,
    "ac3": solve_sudoku_with_ac3
}

# Measure performance
performance_df, detailed_results = step_measure_sudoku_performance_with_heuristics(["easy", "medium", "hard"], heuristics)

# Display the performance DataFrame
print(performance_df)

# Access and display detailed results
for result in detailed_results:
    print("\nHeuristic:", result["Heuristic"])
    print("Difficulty:", result["Difficulty"])
    print("Total Steps Taken:", result["Total Steps"])
    print("Initial Grid:")
    print_board_framed(result["Initial Grid"])
    print("First 5 Sudoku States:")
    for i, snapshot in enumerate(result["First 5 Steps"], start=1):
        print(f"After Step {i}:")
        print_board_framed(snapshot)
    if result["Solved Grid"]:
        print("Solved Grid:")
        print_board_framed(result["Solved Grid"])
    else:
        print("No solution found.")
    print("Time Taken (s):", result["Time Taken (s)"])
    print("-" * 50)

   Heuristic Difficulty  Time Taken (s)  Solved  Total Steps
0        mrv       easy          0.0112    True         6163
2        mrv       hard          0.0156    True         8369
1        mrv     medium          0.0157    True         7951
9        ac3       easy          0.0166    True         8923
10       ac3     medium          0.0192    True        10437
3        lcv       easy          0.0194    True        10744
6   combined       easy          0.0200    True        11528
4        lcv     medium          0.0230    True        12691
7   combined     medium          0.0278    True        15539
11       ac3       hard          0.0316    True        14558
8   combined       hard          0.0372    True        19845
5        lcv       hard          0.0471    True        26596

Heuristic: mrv
Difficulty: easy
Total Steps Taken: 6163
Initial Grid:
-------+-------+-------
1 2 . | . . . | 7 . .
. 5 . | 7 8 9 | 1 2 .
7 . 9 | 1 2 3 | . . 6
-------+-------+-------
2 3 . | . 7 4 | 8 . .


In [20]:
def sudoku_solver_algo(difficulties, heuristics, grid_size=9, user_grid=None):
    """
    Measure the performance of various heuristics on a Sudoku grid.
    Allows the user to provide a Sudoku grid instead of generating one.
    Returns the initial grid, first 5 steps, solved grid, and performance metrics.
    """
    results = []
    detailed_results = []  # To store initial grid, solved grid, and other details

    def solver_with_steps(solver, board, grid_size, subgrid_size):
        """Wrapper to capture first 5 steps and their resulting Sudoku grids."""
        steps = []
        snapshots = []  # To store the board state after each step
        total_steps = 0

        def track_step(row, col, value):
            nonlocal total_steps
            total_steps += 1
            if len(snapshots) < 5:
                steps.append((row, col, value))
                # Capture the current state of the board
                snapshots.append([row[:] for row in board])

        original_solve = solver

        def wrapped_solver(board, grid_size, subgrid_size):
            for row in range(grid_size):
                for col in range(grid_size):
                    if board[row][col] == 0:
                        for num in range(1, grid_size + 1):
                            if is_valid(board, row, col, num, grid_size, subgrid_size):
                                board[row][col] = num
                                track_step(row, col, num)
                                if wrapped_solver(board, grid_size, subgrid_size):
                                    return True
                                board[row][col] = 0
                        return False
            return True

        solved = wrapped_solver(board, grid_size, subgrid_size)
        return solved, snapshots, total_steps


    for heuristic_name, solver in heuristics.items():
        for difficulty in difficulties:
            # Use user-provided grid if available; otherwise, generate one
            if user_grid:
                initial_grid = [row[:] for row in user_grid]
            else:
                initial_grid = generate_full_grid(grid_size, int(math.sqrt(grid_size)))
                remove_numbers_from_grid(
                    initial_grid,
                    {"easy": grid_size**2 // 3, "medium": grid_size**2 // 2, "hard": 2 * grid_size**2 // 3}[difficulty]
                )
            
            # Copy the grid for solving
            sudoku_grid = [row[:] for row in initial_grid]

            start_time = time.time()
            solved, snapshots, total_steps = solver_with_steps(solver, sudoku_grid, grid_size, int(math.sqrt(grid_size)))
            end_time = time.time()

            # Append performance results
            results.append({
                "Grid Size": grid_size,
                "Heuristic": heuristic_name,
                "Difficulty": difficulty,
                "Time Taken (s)": round(end_time - start_time, 4),
                "Solved": solved,  # True if the puzzle is solved, False otherwise
                "Total Steps": total_steps
            })

            # Append detailed results
            detailed_results.append({
                "Heuristic": heuristic_name,
                "Difficulty": difficulty,
                "Initial Grid": initial_grid,
                "First 5 Steps": snapshots,
                "Solved Grid": sudoku_grid if solved else None,
                "Time Taken (s)": round(end_time - start_time, 4),
                "Total Steps": total_steps
            })

    # Create DataFrame for performance
    performance_df = pd.DataFrame(results).sort_values(by="Time Taken (s)", ascending=True)

    return performance_df, detailed_results

# Example Usage:
heuristics = {
    "ac3": solve_sudoku_with_ac3,
    "mrv": solve_sudoku,
    "lcv": lambda b, g, s: solve_sudoku(b, g, s, "lcv"),
    "combined": solve_sudoku_combined,
    "ac3": solve_sudoku_with_ac3
}


# Define your custom grid (replace with your own grid)
initial_grid = [
    [9, 1, 3, 0, 0, 0, 5, 0, 0],
    [6, 0, 7, 0, 0, 0, 0, 2, 4],
    [0, 5, 0, 0, 8, 0, 0, 7, 0],
    [0, 7, 9, 0, 0, 0, 0, 0, 0],
    [0, 0, 2, 0, 9, 0, 0, 4, 3],
    [0, 0, 0, 0, 0, 4, 0, 9, 0],
    [0, 4, 0, 0, 0, 1, 9, 0, 0],
    [7, 0, 6, 0, 0, 9, 0, 0, 5],
    [0, 0, 1, 0, 0, 6, 4, 0, 7]
]

# Measure performance with your custom grid
performance_df, detailed_results = sudoku_solver_algo(["easy"], heuristics, user_grid=initial_grid)

# Display the performance DataFrame
print(performance_df)

# Access and display detailed results
for result in detailed_results:
    print("\nHeuristic:", result["Heuristic"])
    print("Difficulty:", result["Difficulty"])
    print("Initial Grid:")
    print_board_framed(result["Initial Grid"])
    print("First 5 Sudoku States:")
    for i, snapshot in enumerate(result["First 5 Steps"], start=1):
        print(f"After Step {i}:")
        print_board_framed(snapshot)
    if result["Solved Grid"]:
        print("Solved Grid:")
        print_board_framed(result["Solved Grid"])
    else:
        print("No solution found.")
    print("Time Taken (s):", result["Time Taken (s)"])
    print("-" * 50)
    
    

   Grid Size Heuristic Difficulty  Time Taken (s)  Solved  Total Steps
2          9       lcv       easy          0.0319    True         2370
1          9       mrv       easy          0.0323    True         2370
0          9       ac3       easy          0.0326    True         2370
3          9  combined       easy          0.0334    True         2370

Heuristic: ac3
Difficulty: easy
Initial Grid:
-------+-------+-------
9 1 3 | . . . | 5 . .
6 . 7 | . . . | . 2 4
. 5 . | . 8 . | . 7 .
-------+-------+-------
. 7 9 | . . . | . . .
. . 2 | . 9 . | . 4 3
. . . | . . 4 | . 9 .
-------+-------+-------
. 4 . | . . 1 | 9 . .
7 . 6 | . . 9 | . . 5
. . 1 | . . 6 | 4 . 7
-------+-------+-------
First 5 Sudoku States:
After Step 1:
-------+-------+-------
9 1 3 | 2 . . | 5 . .
6 . 7 | . . . | . 2 4
. 5 . | . 8 . | . 7 .
-------+-------+-------
. 7 9 | . . . | . . .
. . 2 | . 9 . | . 4 3
. . . | . . 4 | . 9 .
-------+-------+-------
. 4 . | . . 1 | 9 . .
7 . 6 | . . 9 | . . 5
. . 1 | . . 6 | 4 .

In [21]:
import random
import time
from collections import deque
import math


# Helper functions
def is_valid(board, row, col, num, grid_size, subgrid_size):
    """Check if placing num in board[row][col] is valid."""
    for i in range(grid_size):
        # Check row
        if board[row][i] == num:
            return False
        # Check column
        if board[i][col] == num:
            return False
        # Check subgrid
        subgrid_row = subgrid_size * (row // subgrid_size)
        subgrid_col = subgrid_size * (col // subgrid_size)
        if board[subgrid_row + i // subgrid_size][subgrid_col + i % subgrid_size] == num:
            return False
    return True


def find_empty_cell(board, grid_size):
    """Find the first empty cell in the grid."""
    for row in range(grid_size):
        for col in range(grid_size):
            if board[row][col] == 0:
                return row, col
    return None


# BFS Solver
def solve_sudoku_bfs(board, grid_size, subgrid_size):
    """Solve Sudoku using Breadth-First Search."""
    queue = deque([board])
    while queue:
        current_board = queue.popleft()
        empty_cell = find_empty_cell(current_board, grid_size)

        if not empty_cell:
            return current_board  # Puzzle solved

        row, col = empty_cell
        for num in range(1, grid_size + 1):
            if is_valid(current_board, row, col, num, grid_size, subgrid_size):
                new_board = [r[:] for r in current_board]
                new_board[row][col] = num
                queue.append(new_board)

    return None  # No solution found


# DFS Solver
def solve_sudoku_dfs(board, grid_size, subgrid_size):
    """Solve Sudoku using Depth-First Search."""
    stack = [board]
    while stack:
        current_board = stack.pop()
        empty_cell = find_empty_cell(current_board, grid_size)

        if not empty_cell:
            return current_board  # Puzzle solved

        row, col = empty_cell
        for num in range(1, grid_size + 1):
            if is_valid(current_board, row, col, num, grid_size, subgrid_size):
                new_board = [r[:] for r in current_board]
                new_board[row][col] = num
                stack.append(new_board)

    return None  # No solution found


# IDS Solver
def solve_sudoku_ids(board, grid_size, subgrid_size):
    """Solve Sudoku using Iterative Deepening Search."""
    def dfs_limited(board, depth):
        """Depth-limited DFS for IDS."""
        empty_cell = find_empty_cell(board, grid_size)
        if not empty_cell:
            return board  # Puzzle solved

        if depth == 0:
            return None

        row, col = empty_cell
        for num in range(1, grid_size + 1):
            if is_valid(board, row, col, num, grid_size, subgrid_size):
                new_board = [r[:] for r in board]
                new_board[row][col] = num
                result = dfs_limited(new_board, depth - 1)
                if result:
                    return result
        return None

    for depth in range(1, grid_size ** 2 + 1):
        result = dfs_limited(board, depth)
        if result:
            return result
    return None  # No solution found


# Generate a valid Sudoku grid
def generate_sudoku(grid_size):
    """Generate a valid Sudoku grid."""
    subgrid_size = int(math.sqrt(grid_size))
    solved_board = [
        [((i * subgrid_size + i // subgrid_size + j) % grid_size) + 1 for j in range(grid_size)]
        for i in range(grid_size)
    ]
    # Shuffle rows and columns within subgrids for randomness
    for i in range(0, grid_size, subgrid_size):
        rows = list(range(i, i + subgrid_size))
        cols = list(range(i, i + subgrid_size))
        random.shuffle(rows)
        random.shuffle(cols)
        solved_board[i:i + subgrid_size] = [solved_board[r] for r in rows]
        for row in solved_board:
            row[i:i + subgrid_size] = [row[c] for c in cols]
    return solved_board


def remove_numbers(board, difficulty, grid_size):
    """Remove numbers from the Sudoku grid based on difficulty."""
    empty_cells = {"easy": grid_size ** 2 // 4, "medium": grid_size ** 2 // 2, "hard": 3 * grid_size ** 2 // 4}[difficulty]
    for _ in range(empty_cells):
        row, col = random.randint(0, grid_size - 1), random.randint(0, grid_size - 1)
        board[row][col] = 0
    return board


# Print the Sudoku board
def print_board(board):
    """Print the Sudoku board in a readable format."""
    for row in board:
        print(" ".join(str(cell) if cell != 0 else "." for cell in row))


# Solve with specified models
def solve_with_models(board, grid_size, subgrid_size, models):
    """Solve Sudoku with specified models and measure times."""
    solutions = {}
    for model_name, model_function in models.items():
        print(f"\nSolving with {model_name}...")
        start_time = time.time()
        solution = model_function(board, grid_size, subgrid_size)
        end_time = time.time()
        solutions[model_name] = {
            "solution": solution,
            "time": round(end_time - start_time, 4)
        }
    return solutions


# Main function
if __name__ == "__main__":
    grid_size = 9  # Change grid size (e.g., 9 for 9x9 Sudoku)
    difficulty = "hard"  # Choose difficulty: "easy", "medium", "hard"
    subgrid_size = int(math.sqrt(grid_size))

    # Generate and prepare Sudoku grid
    sudoku_grid = generate_sudoku(grid_size)
    sudoku_grid = remove_numbers(sudoku_grid, difficulty, grid_size)

    print("Original Sudoku Grid:")
    print_board(sudoku_grid)

    # Specify solving models
    models = {
        "BFS": solve_sudoku_bfs,
        "DFS": solve_sudoku_dfs,
        "IDS": solve_sudoku_ids
    }

    # Solve and display results
    results = solve_with_models(sudoku_grid, grid_size, subgrid_size, models)
    for model, result in results.items():
        print(f"\n{model} Solution:")
        if result["solution"]:
            print_board(result["solution"])
            print(f"Time Taken ({model}): {result['time']} seconds")
        else:
            print(f"No solution found with {model}.")


Original Sudoku Grid:
3 . . . . 6 . . .
6 4 5 7 . 9 1 . 3
. . . 1 . 3 . . .
. . . 2 3 . . . 7
4 . . . 6 7 8 . 1
. 5 6 8 9 . 2 . 4
2 . . . . 5 6 . 8
8 . . 9 . 2 . . .
. 3 . . 7 8 . . 2

Solving with BFS...

Solving with DFS...

Solving with IDS...

BFS Solution:
3 1 2 4 5 6 7 8 9
6 4 5 7 8 9 1 2 3
9 7 8 1 2 3 4 5 6
1 8 9 2 3 4 5 6 7
4 2 3 5 6 7 8 9 1
7 5 6 8 9 1 2 3 4
2 9 1 3 4 5 6 7 8
8 6 7 9 1 2 3 4 5
5 3 4 6 7 8 9 1 2
Time Taken (BFS): 0.0787 seconds

DFS Solution:
3 8 1 4 5 6 7 2 9
6 4 5 7 2 9 1 8 3
9 7 2 1 8 3 4 5 6
1 9 8 2 3 4 5 6 7
4 2 3 5 6 7 8 9 1
7 5 6 8 9 1 2 3 4
2 1 9 3 4 5 6 7 8
8 6 7 9 1 2 3 4 5
5 3 4 6 7 8 9 1 2
Time Taken (DFS): 0.0093 seconds

IDS Solution:
3 1 2 4 5 6 7 8 9
6 4 5 7 8 9 1 2 3
9 7 8 1 2 3 4 5 6
1 8 9 2 3 4 5 6 7
4 2 3 5 6 7 8 9 1
7 5 6 8 9 1 2 3 4
2 9 1 3 4 5 6 7 8
8 6 7 9 1 2 3 4 5
5 3 4 6 7 8 9 1 2
Time Taken (IDS): 1.8757 seconds
